# Facial Keypoint Detection — Evaluación (Local, DirectML)

Ejecuta después de `train.ipynb`.


## 1. Imports

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

try:
    import torch_directml
    device = torch_directml.device()
    print(f'Usando DirectML (AMD GPU): {device}')
except ImportError:
    device = torch.device('cpu')
    print('torch-directml no encontrado, usando CPU.')


## 2. Configuración

In [ ]:
# ⚠️ Ajusta esta ruta
DATA_DIR       = r'F:\DL_Project\Data'
IMAGES_PATH    = os.path.join(DATA_DIR, 'preprocessed', 'images_50k.npy')
KEYPOINTS_PATH = os.path.join(DATA_DIR, 'preprocessed', 'keypoints_50k.npy')
MODEL_PATH     = os.path.join(DATA_DIR, 'best_baseline_cnn.pth')
LOSSES_PATH    = os.path.join(DATA_DIR, 'training_losses.pkl')

IMG_SIZE   = 96
BATCH_SIZE = 64
SEED       = 42
N_SHOW     = 8


## 3. Cargar datos y modelo

In [ ]:
class FacialKeypointDataset(Dataset):
    def __init__(self, images, keypoints):
        self.images    = torch.tensor(images,    dtype=torch.float32)
        self.keypoints = torch.tensor(keypoints, dtype=torch.float32)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.keypoints[idx]


class BaselineCNN(nn.Module):
    def __init__(self, num_keypoints=68):
        super().__init__()
        out_dim = num_keypoints * 2
        self.features = nn.Sequential(
            nn.Conv2d(3,   32, kernel_size=3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32,  64, kernel_size=3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128,256, kernel_size=3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 1024), nn.ReLU(inplace=True),
            nn.Linear(1024, 256),          nn.ReLU(inplace=True),
            nn.Linear(256, out_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.regressor(self.features(x))


# Cargar datos
print('Cargando datos...')
images    = np.load(IMAGES_PATH)
keypoints = np.load(KEYPOINTS_PATH)
print(f'images:    {images.shape}')
print(f'keypoints: {keypoints.shape}')

# Reproducir exactamente el mismo val split que en train.ipynb
full_dataset = FacialKeypointDataset(images, keypoints)
torch.manual_seed(SEED)
train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
_, val_dataset = random_split(full_dataset, [train_size, val_size])

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'\nVal: {len(val_dataset):,} frames  ({len(val_loader)} batches)')

# Cargar modelo
print(f'\nCargando modelo desde: {MODEL_PATH}')
model = BaselineCNN()
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model = model.to(device)
model.eval()
print('Modelo cargado. ✅')


## 4. Curva de pérdida

In [ ]:
with open(LOSSES_PATH, 'rb') as f:
    losses = pickle.load(f)

epochs = range(1, len(losses['train']) + 1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs, losses['train'], label='Train loss',      color='steelblue')
ax.plot(epochs, losses['val'],   label='Validation loss', color='tomato')
ax.set_xlabel('Época'); ax.set_ylabel('MSE Loss')
ax.set_title('Curva de pérdida — CNN Baseline')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 5. RMSE en píxeles

In [ ]:
model.eval()
all_gt, all_pred = [], []

with torch.no_grad():
    for imgs, kps in val_loader:
        imgs = imgs.to(device)
        preds = model(imgs).cpu().numpy()
        all_gt.append(kps.numpy())
        all_pred.append(preds)

all_gt   = np.concatenate(all_gt,   axis=0) * IMG_SIZE
all_pred = np.concatenate(all_pred, axis=0) * IMG_SIZE

diff    = (all_gt - all_pred).reshape(-1, 68, 2)
dist_px = np.sqrt((diff ** 2).sum(axis=2))

rmse_per_kp  = dist_px.mean(axis=0)
rmse_overall = dist_px.mean()

print(f'RMSE global:            {rmse_overall:.2f} px')
print(f'RMSE mínimo (keypoint): {rmse_per_kp.min():.2f} px  (kp #{rmse_per_kp.argmin()})')
print(f'RMSE máximo (keypoint): {rmse_per_kp.max():.2f} px  (kp #{rmse_per_kp.argmax()})')

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(68), rmse_per_kp, color='steelblue', alpha=0.8)
ax.axhline(rmse_overall, color='tomato', linestyle='--', label=f'Media: {rmse_overall:.2f} px')
ax.set_xlabel('Keypoint index'); ax.set_ylabel('RMSE (px)')
ax.set_title('Error por keypoint — CNN Baseline')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## 6. Predicciones visuales

In [ ]:
model.eval()
imgs_b, kps_b = next(iter(val_loader))
imgs_b = imgs_b.to(device)

with torch.no_grad():
    preds_b = model(imgs_b)

imgs_np  = imgs_b.cpu().numpy()[:N_SHOW]
kps_gt   = kps_b.numpy()[:N_SHOW]
kps_pred = preds_b.cpu().numpy()[:N_SHOW]

cols = 4
rows = (N_SHOW + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
axes = axes.flatten()

for i in range(N_SHOW):
    img  = imgs_np[i].transpose(1, 2, 0)
    gt   = kps_gt[i].reshape(-1, 2)   * IMG_SIZE
    pred = kps_pred[i].reshape(-1, 2) * IMG_SIZE

    axes[i].imshow(img)
    axes[i].scatter(gt[:, 0],   gt[:, 1],   s=12, c='lime', zorder=5)
    axes[i].scatter(pred[:, 0], pred[:, 1], s=12, c='red',  zorder=6)
    axes[i].axis('off')

patch_gt   = mpatches.Patch(color='lime', label='Real')
patch_pred = mpatches.Patch(color='red',  label='Predicho')
fig.legend(handles=[patch_gt, patch_pred], loc='lower center',
           ncol=2, fontsize=11, bbox_to_anchor=(0.5, -0.02))
plt.suptitle('Predicciones vs. Ground Truth — CNN Baseline', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()


## 7. Resumen final

In [ ]:
print('=' * 50)
print('RESUMEN FINAL — CNN Baseline')
print('=' * 50)
print(f'Frames usados:    50,000  (train: 40,000  val: 10,000)')
print(f'Épocas:           {len(losses["train"])}')
print(f'Mejor val MSE:    {min(losses["val"]):.6f}')
print(f'RMSE global:      {rmse_overall:.2f} px')
print(f'Modelo:           {MODEL_PATH}')
